# 03 - CatBoost, tuning, more features, seed averaging

**Starting point:** v2 (tree vote) scored **0.80** on Kaggle. Check on train was 0.811.

Four ideas tested here:
1. **More features:** group spending, luxury vs basic spending, cabin mates, children spend nothing.
2. **CatBoost:** a tree model that handles text columns (planet, deck ...) directly.
3. **Automatic tuning (Optuna):** try many LightGBM settings and keep the best.
4. **Blend + seed averaging:** average CatBoost and LightGBM, each trained with several random seeds.

**Honest checking:** every idea is checked on **two different splits** of the train data (split A and split B).
A gain that shows up on only one split is probably luck.

Output: `submission_v3.csv`. Takes about 15 minutes to run (CatBoost is slow).

In [1]:
# --- Libraries ---
import warnings
warnings.filterwarnings("ignore")                     # hide harmless library warnings
import numpy as np
import pandas as pd
import optuna                                         # automatic settings search
from catboost import CatBoostClassifier               # tree model that handles text columns itself
from lightgbm import LGBMClassifier                   # fast tree model
from sklearn.model_selection import StratifiedGroupKFold
optuna.logging.set_verbosity(optuna.logging.WARNING)  # don't print every tuning attempt

# --- Load the data (same as notebooks 01 and 02) ---
train = pd.read_csv("spaceship-titanic-data/train.csv")
test = pd.read_csv("spaceship-titanic-data/test.csv")
df = pd.concat([train, test], ignore_index=True)     # clean train and test the same way
is_train = df.Transported.notna()                     # True = train row

SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
df["Group"] = df.PassengerId.str[:4]                  # travel group number
y = train.Transported.astype(int)                     # answers as 1 / 0
groups = df.loc[is_train, "Group"]

# Two different 5-part splits. Each keeps a travel group together in one part.
SPLIT_A = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
SPLIT_B = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

## 1. Fill gaps (same logic as notebook 02, plus one new rule)
**New rule:** children under 13 can't spend money, so their missing spending is 0.

In [2]:
# Rule: sleepers spend nothing
sleeping = df.CryoSleep == True
df.loc[sleeping, SPEND] = df.loc[sleeping, SPEND].fillna(0)

# NEW rule: children under 13 spend nothing (in train, they never spend)
child = df.Age < 13
print("Money spent by children in train:", df.loc[is_train & child, SPEND].sum().sum())
df.loc[child, SPEND] = df.loc[child, SPEND].fillna(0)

# Rule: anyone who spent money was awake
spent = df[SPEND].sum(axis=1, min_count=1)
df.loc[df.CryoSleep.isna() & (spent > 0), "CryoSleep"] = False

# Rule: same group / same surname -> same home planet
df["Surname"] = df.Name.str.split().str[-1]
for key in ["Group", "Surname"]:
    planet = df.dropna(subset=["HomePlanet"]).groupby(key).HomePlanet.agg(lambda s: s.mode()[0])
    df["HomePlanet"] = df.HomePlanet.fillna(df[key].map(planet))

# Rule: split Cabin into deck / number / side; group members share deck and side
cabin = df.Cabin.str.split("/", expand=True)
df["Deck"], df["CabinNum"], df["Side"] = cabin[0], pd.to_numeric(cabin[1]), cabin[2]
for c in ["Deck", "Side"]:
    common = df.dropna(subset=[c]).groupby("Group")[c].agg(lambda s: s.mode()[0])
    df[c] = df[c].fillna(df.Group.map(common))

# Whatever is still missing -> simple values
for c in SPEND:
    df[c] = df[c].fillna(0)
df["CryoSleep"] = df.CryoSleep.fillna(df[SPEND].sum(axis=1) == 0)    # spent nothing -> likely asleep
df["Age"] = df.Age.fillna(df[is_train].Age.median())
df["CabinNum"] = df.CabinNum.fillna(df.CabinNum.median())
for c in ["HomePlanet", "Destination", "Deck", "Side"]:
    df[c] = df[c].fillna(df[c].mode()[0])
df["VIP"] = df.VIP.fillna(False)
print("Gaps left in model columns:", df[SPEND + ["CryoSleep", "Age", "HomePlanet", "Deck", "Side"]].isna().sum().sum())

Money spent by children in train: 0.0


Gaps left in model columns: 0


## 2. Features
Same features as notebook 02, plus the **new** ones below.

In [3]:
# --- Features from notebook 02 ---
df["TotalSpend"] = df[SPEND].sum(axis=1)
df["NoSpend"] = (df.TotalSpend == 0).astype(int)
df["GroupSize"] = df.groupby("Group").Group.transform("size")
df["Alone"] = (df.GroupSize == 1).astype(int)
df["FamilySize"] = df.groupby("Surname").Surname.transform("size").fillna(1)
df["CabinRegion"] = (df.CabinNum // 300).astype(int)
df["Child"] = (df.Age < 13).astype(int)

# --- NEW features ---
df["Luxury"] = df.RoomService + df.Spa + df.VRDeck                    # "luxury" spending
df["Basic"] = df.FoodCourt + df.ShoppingMall                          # "basic" spending
df["GroupSpend"] = df.groupby("Group").TotalSpend.transform("sum")    # whole group's spending
df["GroupCryoShare"] = df.groupby("Group").CryoSleep.transform("mean").astype(float)  # share of group asleep
df["SpendVsGroup"] = df.TotalSpend / (df.GroupSpend + 1)              # my share of the group's spending
df["CabinMates"] = df.groupby(["Deck", "CabinNum", "Side"]).Group.transform("size")   # people in my cabin

OLD_NUM = ["Age", "NoSpend", "GroupSize", "Alone", "FamilySize", "CabinNum", "CabinRegion", "Child"]
NEW_NUM = ["GroupCryoShare", "SpendVsGroup", "CabinMates"]
OLD_MONEY = SPEND + ["TotalSpend"]
NEW_MONEY = ["Luxury", "Basic", "GroupSpend"]
TEXT = ["HomePlanet", "Destination", "Deck", "Side"]

def make_table(new_features, text_as_columns=True):
    """Build the table of numbers for a model.
    new_features      : include the NEW features or not
    text_as_columns   : True  -> one 0/1 column per text value (for LightGBM)
                        False -> keep text as it is (CatBoost handles text itself)"""
    money = OLD_MONEY + (NEW_MONEY if new_features else [])
    nums = OLD_NUM + (NEW_NUM if new_features else [])
    X = np.log1p(df[money])                               # log1p shrinks big money amounts
    X = X.join(df[nums])
    X["CryoSleep"] = df.CryoSleep.astype(int)
    X["VIP"] = df.VIP.astype(int)
    if text_as_columns:
        for c in TEXT:
            X = X.join(pd.get_dummies(df[c], prefix=c).astype(int))
    else:
        X = X.join(df[TEXT])
    return X

X_old = make_table(new_features=False)           # notebook 02 features
X_new = make_table(new_features=True)            # with the new features
X_cat = make_table(new_features=True, text_as_columns=False)   # for CatBoost
print("Columns: old", X_old.shape[1], "| new", X_new.shape[1], "| CatBoost", X_cat.shape[1])

Columns: old 32 | new 38 | CatBoost 26


## Helper: score a model on a split
For each of the 5 parts, train on the other 4 and predict this part.
This gives every train passenger a prediction from a model that never saw them ("out-of-fold").
We keep the predicted **chances** (0 to 1), so we can average models later.

In [4]:
def out_of_fold(make_model, X, split):
    """Returns each train passenger's predicted chance of being transported."""
    X = X[is_train].reset_index(drop=True)
    chance = np.zeros(len(X))
    for train_rows, check_rows in split.split(X, y, groups):
        model = make_model()                                  # fresh model each time
        model.fit(X.iloc[train_rows], y.iloc[train_rows])
        chance[check_rows] = model.predict_proba(X.iloc[check_rows])[:, 1]
    return chance

def accuracy(chance):
    """Chance above 0.5 -> predict transported. Share of correct answers."""
    return ((chance > 0.5).astype(int) == y.values).mean()

# The LightGBM settings used in notebook 02
def lgbm_default():
    return LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=20, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.7, random_state=0, verbose=-1)

results = {}    # we fill this table as we go: idea -> accuracy on split A and B

## Idea 1: Do the new features help? (LightGBM)

In [5]:
for name, X in [("LightGBM, old features (v2)", X_old), ("LightGBM, new features", X_new)]:
    results[name] = [accuracy(out_of_fold(lgbm_default, X, s)) for s in (SPLIT_A, SPLIT_B)]
pd.DataFrame(results, index=["Split A", "Split B"]).T.round(4)

,Split A,Split B
"LightGBM, old features (v2)",0.8115,0.8135
"LightGBM, new features",0.8117,0.8140


**Almost no change.** Tree models can already work out group spending from the existing columns.
The new features are kept (they don't hurt), but they are not where the gain comes from.

## Idea 2: CatBoost
CatBoost reads text columns (planet, deck ...) directly, instead of 0/1 columns. Slow: about 4 minutes per split.

In [6]:
def catboost_model(seed=0):
    # iterations = number of trees, depth = how complex each tree is
    return CatBoostClassifier(iterations=1000, learning_rate=0.03, depth=6, random_seed=seed,
                              cat_features=TEXT, verbose=0)

# Keep the chances: we need them again for the blend in idea 4
chance_cat = {s: out_of_fold(catboost_model, X_cat, split) for s, split in [("A", SPLIT_A), ("B", SPLIT_B)]}
results["CatBoost, new features"] = [accuracy(chance_cat["A"]), accuracy(chance_cat["B"])]
pd.DataFrame(results, index=["Split A", "Split B"]).T.round(4)

,Split A,Split B
"LightGBM, old features (v2)",0.8115,0.8135
"LightGBM, new features",0.8117,0.8140
"CatBoost, new features",0.8162,0.8131


## Idea 3: Automatic tuning with Optuna
Optuna tries 30 different LightGBM settings on **split A** and keeps the best.
Then we check the winner on **split B**. Settings tuned on one split often look good only on that split.

In [7]:
def tuning_score(trial):
    # Optuna picks a value for each setting from these ranges
    settings = dict(
        n_estimators=trial.suggest_int("n_estimators", 200, 1200),           # number of trees
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),                   # tree complexity
        min_child_samples=trial.suggest_int("min_child_samples", 10, 100),   # min passengers per leaf
        subsample=trial.suggest_float("subsample", 0.5, 1.0),                # share of rows per tree
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),  # share of columns per tree
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),    # regularization strength
    )
    return accuracy(out_of_fold(lambda: LGBMClassifier(**settings, subsample_freq=1, random_state=0, verbose=-1),
                                X_new, SPLIT_A))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(tuning_score, n_trials=30)
BEST = study.best_params
print("Best settings found:", {k: round(v, 3) for k, v in BEST.items()})

def lgbm_tuned(seed=0):
    return LGBMClassifier(**BEST, subsample_freq=1, random_state=seed, verbose=-1)

chance_lgbm = {s: out_of_fold(lgbm_tuned, X_new, split) for s, split in [("A", SPLIT_A), ("B", SPLIT_B)]}
results["LightGBM, tuned"] = [accuracy(chance_lgbm["A"]), accuracy(chance_lgbm["B"])]
pd.DataFrame(results, index=["Split A", "Split B"]).T.round(4)

Best settings found: {'n_estimators': 768, 'learning_rate': 0.01, 'num_leaves': 49, 'min_child_samples': 76, 'subsample': 0.584, 'colsample_bytree': 0.611, 'reg_lambda': 0.015}


,Split A,Split B
"LightGBM, old features (v2)",0.8115,0.8135
"LightGBM, new features",0.8117,0.8140
"CatBoost, new features",0.8162,0.8131
"LightGBM, tuned",0.8131,0.8142


Tuning looks better on split A (where it was tuned), but the gain mostly disappears on split B.
**So tuning doesn't really help here.** The default settings were already fine.

## Idea 4: Blend CatBoost + LightGBM
Average the two models' chances. Their mistakes are partly different, so averaging can cancel some out.

In [8]:
# Reuse the chances saved in ideas 2 and 3 (no re-training needed), average them, then score
results["Blend: CatBoost + LightGBM"] = [accuracy((chance_cat[s] + chance_lgbm[s]) / 2) for s in ["A", "B"]]

table = pd.DataFrame(results, index=["Split A", "Split B"]).T
table["Average"] = table.mean(axis=1)
table.round(4)

,Split A,Split B,Average
"LightGBM, old features (v2)",0.8115,0.8135,0.8125
"LightGBM, new features",0.8117,0.8140,0.8128
"CatBoost, new features",0.8162,0.8131,0.8146
"LightGBM, tuned",0.8131,0.8142,0.8136
Blend: CatBoost + LightGBM,0.8151,0.8156,0.8154


## Final model: blend + seed averaging
- Train CatBoost and LightGBM on **all** train data.
- Train each one **3 times** with a different random seed and average. This makes predictions steadier.
- Average CatBoost and LightGBM, then predict transported if the chance is above 0.5.

In [9]:
X_cat_train, X_cat_test = X_cat[is_train], X_cat[~is_train]
X_new_train, X_new_test = X_new[is_train], X_new[~is_train]

SEEDS = [0, 1, 2]
cat_chance = np.mean([catboost_model(s).fit(X_cat_train, y).predict_proba(X_cat_test)[:, 1] for s in SEEDS], axis=0)
lgbm_chance = np.mean([lgbm_tuned(s).fit(X_new_train, y).predict_proba(X_new_test)[:, 1] for s in SEEDS], axis=0)
final_chance = (cat_chance + lgbm_chance) / 2

# Kaggle wants True / False
sub = pd.DataFrame({"PassengerId": test.PassengerId, "Transported": final_chance > 0.5})
sub.to_csv("submission_v3.csv", index=False)

v2 = pd.read_csv("submission_v2.csv")
print(len(sub), "rows,", f"{sub.Transported.mean():.0%} predicted transported")
print("Different from v2:", (sub.Transported.values != v2.Transported.values).sum(), "passengers")

4277 rows, 51% predicted transported
Different from v2: 182 passengers


## Summary
See the table above (idea 4) for the exact numbers.

- **New features:** no real gain.
- **CatBoost:** clearly ahead of LightGBM on split A (+0.45%), slightly behind on split B. About even overall.
- **Tuning:** gain on the split it was tuned on mostly disappears on the other split.
- **Blend:** best average (0.815) and the only option that is good on **both** splits. Used for v3, with seed averaging on top.

Expect Kaggle to land **about 1% below** the train check, like v2 did (0.811 → 0.80).
Gains this small (0.1–0.5%) can be hidden by luck on Kaggle's test set.